# 문서 타입 분류 — 실험 노트북

**프로젝트 구조:**
```
project/
├── src/
|    ├── 00_eda.ipynb         # EDA
|    ├── 01_main.ipynb        # 실험 노트북 (K-FOLD, Split, 모델 비교)
|    ├── 02_main_total.ipynb  # 전체데이터 학습, 추론, 제출 
|    ├── config.py            # 하이퍼파라미터 & 경로 
|    ├── preprocess.py        # 오프라인 증강 함수
|    ├── augmentation.py      # 온라인 증강 (v1/v2/v3 전환) & TTA
|    ├── dataset.py           # Dataset, MixUp, CutMix
|    ├── train.py             # 학습/검증 함수
|    ├── inference.py         # TTA 추론, 앙상블 
├── data/
├── outputs/
├── requirements.txt
```

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

import importlib
import config
importlib.reload(config)
from config import *
import train
importlib.reload(train)

import augmentation
importlib.reload(augmentation)
from augmentation import get_train_transforms
from dataset import DocDataset
from train import build_model, get_class_weights, train_one_epoch #,FocalLoss
from inference import predict_tta, save_submission, ensemble_from_files

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])
print(f'Device: {CFG["device"]}')
print(f'Model: {CFG["model_name"]} | Size: {CFG["img_size"]}')
print(f'use_mixup={CFG["use_mixup"]}, use_cutmix={CFG["use_cutmix"]}')  # 적용된 config값 확인

---
## 1. 데이터 로드

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_meta  = pd.read_csv(META_CSV)
df_test  = pd.read_csv(SAMPLE_CSV)
target_to_name = dict(zip(df_meta['target'], df_meta['class_name']))

print(f'Train: {len(df_train)}장 | Test: {len(df_test)}장')

# 클래스 분포 확인
class_counts = df_train['target'].value_counts().sort_index()
ratio = class_counts.max() / class_counts.min()
print(f'\n불균형 비율: {ratio:.2f}x (max={class_counts.max()} / min={class_counts.min()})')
print(class_counts)

---
## 2. WandB 초기화

In [ ]:
if CFG['use_wandb']:
    import wandb
    from dotenv import load_dotenv
    
    load_dotenv(PROJECT_ROOT / '.env')  
    wandb.login(key=os.environ.get('WANDB_API_KEY'))

    run_name = CFG['wandb_run_name'] or f"{CFG['model_name']}_augv2_tta10"
    wandb.init(
        project=CFG['wandb_project'],
        name=run_name,
        config=CFG,
        tags=[CFG['model_name'], 'augv2_tta10'],
        settings=wandb.Settings(silent=True, _disable_stats=True),
    )
    print(f'WandB: {CFG["wandb_project"]} / {run_name}')
else:
    print('WandB 비활성화')

---
## 3. K-Fold 검증

In [ ]:
class_weights = get_class_weights(df_train, CFG['num_classes'], CFG['device'])
criterion = nn.CrossEntropyLoss(weight=class_weights)
# criterion = FocalLoss(alpha=class_weights, gamma=2.0)

from train import run_kfold
fold_scores = run_kfold(df_train, TRAIN_DIR, CFG, class_weights, n_folds=5)

---
## 4. 학습

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from augmentation import get_valid_transforms

# ============================================
# 1) Train / Valid 분리 (9:1)
# ============================================
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=CFG['seed'])
train_idx, val_idx = next(sss.split(df_train, df_train['target']))
df_tr = df_train.iloc[train_idx]
df_val = df_train.iloc[val_idx]
print(f'Train: {len(df_tr)}장 | Valid: {len(df_val)}장')

# # ============================================
# # 2) Train만 오프라인 증강 
# # ============================================
# AUG_TRAIN_DIR = DATA_DIR / 'train_doc_aug'
# df_tr_aug = oversample_with_doc_aug(df_tr, TRAIN_DIR, AUG_TRAIN_DIR,
#                                      target_count=100, extra_per_class=50)
# print(f'증강 후 Train: {len(df_tr_aug)}장 | Valid: {len(df_val)}장 (원본)')

# ============================================
# 2) DataLoader
# ============================================
train_ds = DocDataset(df_tr, TRAIN_DIR, # df_tr_aug, AUG_TRAIN_DIR,
                      get_train_transforms(CFG['img_size'], version='v2'))
val_ds   = DocDataset(df_val, TRAIN_DIR,
                      get_valid_transforms(CFG['img_size']))

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'],
                          pin_memory=True)

# ============================================
# 3) 모델 & Loss & Optimizer
# ============================================
model = build_model(CFG['model_name'], CFG['num_classes'],
                    CFG['pretrained']).to(CFG['device'])
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{CFG["model_name"]}: {n_params:.1f}M params')

class_weights = get_class_weights(df_tr, CFG['num_classes'], CFG['device'])
# # 오프라인 증강 후 데이터 기준으로 class weight 계산
# class_weights = get_class_weights(df_tr_aug, CFG['num_classes'], CFG['device'])
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f'Weighted CE')

# # focal loss
# criterion = FocalLoss(alpha=class_weights, gamma=2.0)
# print(f'FocalLoss (gamma=2.0 + class_weight, 불균형 {ratio:.2f}x 보정)')

optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6)

In [ ]:
from train import validate
from sklearn.metrics import confusion_matrix, classification_report

model_tag = CFG['model_name'].replace('.', '_').replace('/', '_')
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
best_val_f1 = 0.0
class_names = [target_to_name[i][:18] for i in range(17)]

print('=' * 60)
print(f'{CFG["model_name"]} | Train {len(df_tr)}장 + Valid {len(df_val)}장 | {CFG["epochs"]}ep')
# print(f'{CFG["model_name"]} | Train {len(df_tr_aug)}장 (증강) + Valid {len(df_val)}장 | {CFG["epochs"]}ep')
print('=' * 60)

for epoch in range(CFG['epochs']):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, CFG)
    val_loss, val_f1 = validate(model, val_loader, criterion, CFG['device'])

    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)

    lr = optimizer.param_groups[0]['lr']
    marker = 'Best!' if val_f1 > best_val_f1 else ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / f'{model_tag}_best.pth')

    print(f'  Epoch {epoch+1:2d}/{CFG["epochs"]} │ '
          f'Train L={train_loss:.4f} F1={train_f1:.4f} │ '
          f'Val L={val_loss:.4f} F1={val_f1:.4f}{marker}')

    if CFG['use_wandb']:
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss, 'train_f1': train_f1,
            'val_loss': val_loss, 'val_f1': val_f1,
            'lr': lr,
        })

print(f'\nBest Val F1: {best_val_f1:.4f}')

# === Best 모델 로드 → Confusion Matrix ===
model.load_state_dict(torch.load(OUTPUT_DIR / f'{model_tag}_best.pth'))
model.eval()
val_preds, val_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(CFG['device']))
        val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        val_labels.extend(labels.numpy())

cm = confusion_matrix(val_labels, val_preds)

# 시각화
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(17)); ax.set_yticks(range(17))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — Best Val F1={best_val_f1:.4f}', fontsize=13)

for i in range(17):
    for j in range(17):
        if cm[i, j] > 0:
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=7, color=color)

plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'confusion_matrix_{model_tag}.png', dpi=150)
plt.show()

# WandB에 confusion matrix 로깅
if CFG['use_wandb']:
    wandb.log({
        'confusion_matrix': wandb.plot.confusion_matrix(
            y_true=val_labels, preds=val_preds, class_names=class_names),
        'confusion_matrix_img': wandb.Image(
            str(OUTPUT_DIR / f'confusion_matrix_{model_tag}.png')),
        'best_val_f1': best_val_f1,
    })

# Classification Report
print('\nClassification Report:')
print(classification_report(val_labels, val_preds,
                            target_names=class_names, digits=4))

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Valid')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['train_f1'], label='Train')
axes[1].plot(history['val_f1'], label='Valid')
axes[1].set_title('F1'); axes[1].legend()
plt.suptitle(CFG['model_name'], fontweight='bold')
plt.tight_layout(); plt.show()

# GPU 정리
del val_loader
torch.cuda.empty_cache()

In [ ]:
# === 노이즈 라벨 탐지 ===
from augmentation import get_valid_transforms

check_ds = DocDataset(df_train, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
check_loader = DataLoader(check_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'])

all_preds, all_probs = [], []
with torch.no_grad():
    for images, labels in tqdm(check_loader, desc='노이즈 탐지'):
        outputs = model(images.to(CFG['device']))
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.array(all_preds)
true_labels = df_train['target'].values

wrong_mask = all_preds != true_labels
wrong_df = df_train[wrong_mask].copy()
wrong_df['pred'] = all_preds[wrong_mask]
wrong_df['confidence'] = all_probs[wrong_mask].max(axis=1)

print(f'오분류 샘플: {len(wrong_df)}개 / {len(df_train)}개')
for _, row in wrong_df.sort_values('confidence', ascending=False).iterrows():
    print(f'  ID={row["ID"]} | 라벨={row["target"]}({target_to_name[row["target"]]}) '
          f'→ 예측={row["pred"]}({target_to_name[row["pred"]]}) '
          f'| conf={row["confidence"]:.3f}')

In [ ]:
# ============================================
# Hard Val — Clean vs Test 조건 비교 (Hard Val: Val set을 Test set과 유사하게)
# ============================================
import importlib, augmentation
importlib.reload(augmentation)
from augmentation import get_hard_valid_transforms, get_valid_transforms

N_RUNS = 5
val_labels = df_val['target'].values
from sklearn.metrics import f1_score

# Clean Val
val_ds_clean = DocDataset(df_val, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
val_loader_clean = DataLoader(val_ds_clean, batch_size=CFG['batch_size'],
                               shuffle=False, num_workers=2, pin_memory=True)
probs = []
with torch.no_grad():
    for images, labels in val_loader_clean:
        outputs = model(images.to(CFG['device']))
        probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
clean_f1 = f1_score(val_labels, np.argmax(np.concatenate(probs), axis=1), average='macro')

# Hard Val x N_RUNS
hard_f1s = []
for run in range(N_RUNS):
    val_ds_hard = DocDataset(df_val, TRAIN_DIR, get_hard_valid_transforms(CFG['img_size']))
    val_loader_hard = DataLoader(val_ds_hard, batch_size=CFG['batch_size'],
                                  shuffle=False, num_workers=2, pin_memory=True)
    probs = []
    with torch.no_grad():
        for images, labels in val_loader_hard:
            outputs = model(images.to(CFG['device']))
            probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
    hard_f1s.append(f1_score(val_labels, np.argmax(np.concatenate(probs), axis=1), average='macro'))

print(f'Clean Val F1:  {clean_f1:.4f}')
print(f'Hard  Val F1:  {np.mean(hard_f1s):.4f} ± {np.std(hard_f1s):.4f}')

---
## 5. 추가 모델 학습

In [ ]:
import os, random, importlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

import config; importlib.reload(config); from config import *
CFG['use_wandb'] = False
import train; importlib.reload(train)
import augmentation; importlib.reload(augmentation)
from augmentation import get_train_transforms, get_valid_transforms
from dataset import DocDataset
from train import build_model, get_class_weights, train_one_epoch, validate
from inference import predict_tta, save_submission, ensemble_from_files

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])

df_train = pd.read_csv(TRAIN_CSV)
df_meta = pd.read_csv(META_CSV)
df_test = pd.read_csv(SAMPLE_CSV)
target_to_name = dict(zip(df_meta['target'], df_meta['class_name']))

class_counts = df_train['target'].value_counts().sort_index()
ratio = class_counts.max() / class_counts.min()

print(f'준비 완료 | seed={CFG["seed"]} | 불균형={ratio:.2f}x')

In [ ]:
# 모델 선택 (한번에 하나만)
# ConvNeXt-Small 
CFG['model_name'] = 'convnext_small.fb_in22k_ft_in1k'
CFG['img_size'] = 384; CFG['lr'] = 1e-4; CFG['batch_size'] = 12

# # Swin-Small 
# CFG['model_name'] = 'swin_small_patch4_window7_224'
# CFG['img_size'] = 224; CFG['lr'] = 5e-5; CFG['batch_size'] = 32

# Swin-Base
# small -> base : batch_size(32 → 16, 파라미터가 2배라 메모리 더 씀)
# CFG['model_name'] = 'swin_base_patch4_window7_224'
# CFG['img_size'] = 224; CFG['lr'] = 5e-5; CFG['batch_size'] = 16 

# EfficientNetV2-S 
# CFG['model_name'] = 'tf_efficientnetv2_s.in21k_ft_in1k'
# CFG['img_size'] = 384; CFG['lr'] = 1e-4; CFG['batch_size'] = 16

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from augmentation import get_valid_transforms

# Train / Valid 분리 (9:1)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=CFG['seed'])
train_idx, val_idx = next(sss.split(df_train, df_train['target']))
df_tr = df_train.iloc[train_idx]
df_val = df_train.iloc[val_idx]
print(f'Train: {len(df_tr)}장 | Valid: {len(df_val)}장')

# DataLoader
train_ds = DocDataset(df_tr, TRAIN_DIR,
                      get_train_transforms(CFG['img_size'], version='v2')) 
val_ds   = DocDataset(df_val, TRAIN_DIR,
                      get_valid_transforms(CFG['img_size']))
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          shuffle=True, num_workers=CFG['num_workers'],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'],
                          pin_memory=True)

# 모델
model = build_model(CFG['model_name'], CFG['num_classes'],
                    CFG['pretrained']).to(CFG['device'])
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{CFG["model_name"]}: {n_params:.1f}M params')

# Loss — Class Weight 적용
class_weights = get_class_weights(df_train, CFG['num_classes'], CFG['device'])
criterion = nn.CrossEntropyLoss(weight=class_weights)
print(f'Weighted CrossEntropyLoss')

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6)

In [ ]:
from train import validate
from sklearn.metrics import confusion_matrix, classification_report

model_tag = CFG['model_name'].replace('.', '_').replace('/', '_')
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}
best_val_f1 = 0.0
class_names = [target_to_name[i][:18] for i in range(17)]

print('=' * 60)
print(f'{CFG["model_name"]} | Train {len(df_tr)}장 + Valid {len(df_val)}장 | {CFG["epochs"]}ep')
print('=' * 60)

for epoch in range(CFG['epochs']):
    train_loss, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, CFG)
    val_loss, val_f1 = validate(model, val_loader, criterion, CFG['device'])

    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)

    lr = optimizer.param_groups[0]['lr']
    marker = 'Best!' if val_f1 > best_val_f1 else ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / f'{model_tag}_best.pth')

    print(f'  Epoch {epoch+1:2d}/{CFG["epochs"]} │ '
          f'Train L={train_loss:.4f} F1={train_f1:.4f} │ '
          f'Val L={val_loss:.4f} F1={val_f1:.4f}{marker}')

    if CFG['use_wandb']:
        wandb.log({
            'epoch': epoch + 1,
            'train_loss': train_loss, 'train_f1': train_f1,
            'val_loss': val_loss, 'val_f1': val_f1,
            'lr': lr,
        })

print(f'\nBest Val F1: {best_val_f1:.4f}')

# === Best 모델 로드 → Confusion Matrix ===
model.load_state_dict(torch.load(OUTPUT_DIR / f'{model_tag}_best.pth'))
model.eval()
val_preds, val_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images.to(CFG['device']))
        val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        val_labels.extend(labels.numpy())

cm = confusion_matrix(val_labels, val_preds)

# 시각화
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(17)); ax.set_yticks(range(17))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(class_names, fontsize=7)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — Best Val F1={best_val_f1:.4f}', fontsize=13)

for i in range(17):
    for j in range(17):
        if cm[i, j] > 0:
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontsize=7, color=color)

plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'confusion_matrix_{model_tag}.png', dpi=150)
plt.show()

# WandB에 confusion matrix 로깅
if CFG['use_wandb']:
    wandb.log({
        'confusion_matrix': wandb.plot.confusion_matrix(
            y_true=val_labels, preds=val_preds, class_names=class_names),
        'confusion_matrix_img': wandb.Image(
            str(OUTPUT_DIR / f'confusion_matrix_{model_tag}.png')),
        'best_val_f1': best_val_f1,
    })

# Classification Report
print('\nClassification Report:')
print(classification_report(val_labels, val_preds,
                            target_names=class_names, digits=4))

# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Valid')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['train_f1'], label='Train')
axes[1].plot(history['val_f1'], label='Valid')
axes[1].set_title('F1'); axes[1].legend()
plt.suptitle(CFG['model_name'], fontweight='bold')
plt.tight_layout(); plt.show()

# GPU 정리
del val_loader
torch.cuda.empty_cache()

In [ ]:
# === 노이즈 라벨 탐지 ===
from augmentation import get_valid_transforms

check_ds = DocDataset(df_train, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
check_loader = DataLoader(check_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=CFG['num_workers'])

all_preds, all_probs = [], []
with torch.no_grad():
    for images, labels in tqdm(check_loader, desc='노이즈 탐지'):
        outputs = model(images.to(CFG['device']))
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.array(all_preds)
true_labels = df_train['target'].values

wrong_mask = all_preds != true_labels
wrong_df = df_train[wrong_mask].copy()
wrong_df['pred'] = all_preds[wrong_mask]
wrong_df['confidence'] = all_probs[wrong_mask].max(axis=1)

print(f'오분류 샘플: {len(wrong_df)}개 / {len(df_train)}개')
for _, row in wrong_df.sort_values('confidence', ascending=False).iterrows():
    print(f'  ID={row["ID"]} | 라벨={row["target"]}({target_to_name[row["target"]]}) '
          f'→ 예측={row["pred"]}({target_to_name[row["pred"]]}) '
          f'| conf={row["confidence"]:.3f}')

In [ ]:
# ============================================
# Hard Val — ConvNeXt (Hard Val: Val set을 Test set과 유사하게)
# ============================================
import importlib, augmentation
importlib.reload(augmentation)
from augmentation import get_hard_valid_transforms, get_valid_transforms
from sklearn.metrics import f1_score

N_RUNS = 5
val_labels = df_val['target'].values

val_ds_clean = DocDataset(df_val, TRAIN_DIR, get_valid_transforms(CFG['img_size']))
val_loader_clean = DataLoader(val_ds_clean, batch_size=CFG['batch_size'],
                               shuffle=False, num_workers=2, pin_memory=True)
probs = []
with torch.no_grad():
    for images, labels in val_loader_clean:
        outputs = model(images.to(CFG['device']))
        probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
clean_f1 = f1_score(val_labels, np.argmax(np.concatenate(probs), axis=1), average='macro')

hard_f1s = []
for run in range(N_RUNS):
    val_ds_hard = DocDataset(df_val, TRAIN_DIR, get_hard_valid_transforms(CFG['img_size']))
    val_loader_hard = DataLoader(val_ds_hard, batch_size=CFG['batch_size'],
                                  shuffle=False, num_workers=2, pin_memory=True)
    probs = []
    with torch.no_grad():
        for images, labels in val_loader_hard:
            outputs = model(images.to(CFG['device']))
            probs.append(torch.softmax(outputs, dim=1).cpu().numpy())
    hard_f1s.append(f1_score(val_labels, np.argmax(np.concatenate(probs), axis=1), average='macro'))

print(f'  Clean Val F1:  {clean_f1:.4f}')
print(f'  Hard  Val F1:  {np.mean(hard_f1s):.4f} ± {np.std(hard_f1s):.4f}')

---
## 6. 단일 vs 앙상블 - Hard Val

In [ ]:
# ============================================
# 단일 vs 앙상블 — Hard Val 비교 (Hard Val: Val set을 Test set과 유사하게)
# ============================================
import importlib, augmentation
importlib.reload(augmentation)
from augmentation import get_hard_valid_transforms, get_valid_transforms
from train import build_model
from sklearn.metrics import f1_score

val_labels = df_val['target'].values
N_RUNS = 5

models_cfg = [
  {'name': 'efficientnet_b3', 'tag': 'efficientnet_b3',
     'img_size': 384, 'batch_size': 16},
    {'name': 'convnext_small.fb_in22k_ft_in1k',
     'tag': 'convnext_small_fb_in22k_ft_in1k',
     'img_size': 384, 'batch_size': 12},
    # {'name': 'tf_efficientnetv2_s.in21k_ft_in1k',
    #  'tag': 'tf_efficientnetv2_s_in21k_ft_in1k',
    #  'img_size': 384, 'batch_size': 16},
    # {'name': 'swin_small_patch4_window7_224',
    #  'tag': 'swin_small_patch4_window7_224',
    #  'img_size': 224, 'batch_size': 32},
    # {'name': 'swin_base_patch4_window7_224',
    #  'tag': 'swin_base_patch4_window7_224',
    #  'img_size': 224, 'batch_size': 16},
]

# --- Hard Val 기준 단일 + 앙상블 비교 ---
val_probs_hard = {}

for m_cfg in models_cfg:
    model_tmp = build_model(m_cfg['name'], 17, pretrained=False).to(CFG['device'])
    ckpt = OUTPUT_DIR / f'{m_cfg["tag"]}_best.pth'
    model_tmp.load_state_dict(torch.load(ckpt, map_location=CFG['device']))
    model_tmp.eval()

    run_probs = []
    for run in range(N_RUNS):
        ds = DocDataset(df_val, TRAIN_DIR, get_hard_valid_transforms(m_cfg['img_size']))
        loader = DataLoader(ds, batch_size=m_cfg['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)
        p = []
        with torch.no_grad():
            for images, labels in loader:
                outputs = model_tmp(images.to(CFG['device']))
                p.append(torch.softmax(outputs, dim=1).cpu().numpy())
        run_probs.append(np.concatenate(p, axis=0))

    val_probs_hard[m_cfg['tag']] = run_probs
    del model_tmp
    torch.cuda.empty_cache()

# 결과 출력
tags = list(val_probs_hard.keys())
short = {'efficientnet_b3': 'B3',
         'convnext_small_fb_in22k_ft_in1k': 'ConvNeXt',
         'tf_efficientnetv2_s_in21k_ft_in1k': 'V2-S',
         'swin_small_patch4_window7_224': 'SwinS',
         'swin_base_patch4_window7_224': 'SwinB'}

print(f'\n{"="*70}')
print(f'Hard Val 비교 (각 {N_RUNS}회)')
print(f'{"="*70}')

for run_idx in range(N_RUNS):
    ens_probs = np.mean([val_probs_hard[t][run_idx] for t in tags], axis=0)
    ens_f1 = f1_score(val_labels, np.argmax(ens_probs, axis=1), average='macro')
    parts = ' | '.join([f'{short[t]}={f1_score(val_labels, np.argmax(val_probs_hard[t][run_idx], axis=1), average="macro"):.4f}' for t in tags])
    print(f'  Run {run_idx+1}: {parts} | Ens={ens_f1:.4f}')

print(f'\n{"─"*70}')
for t in tags:
    scores = [f1_score(val_labels, np.argmax(val_probs_hard[t][r], axis=1), average='macro') for r in range(N_RUNS)]
    print(f'  {short[t]:<10} 평균: {np.mean(scores):.4f} ± {np.std(scores):.4f}')

ens_scores = [f1_score(val_labels, np.argmax(np.mean([val_probs_hard[t][r] for t in tags], axis=0), axis=1), average='macro') for r in range(N_RUNS)]
print(f'  {"Ensemble":<10} 평균: {np.mean(ens_scores):.4f} ± {np.std(ens_scores):.4f}')

# 2모델 조합도 비교
from itertools import combinations
print(f'\n{"─"*70}')
print(f'  2모델 앙상블 비교:')
for t1, t2 in combinations(tags, 2):
    scores_2 = [f1_score(val_labels, np.argmax(np.mean([val_probs_hard[t1][r], val_probs_hard[t2][r]], axis=0), axis=1), average='macro') for r in range(N_RUNS)]
    print(f'  {short[t1]}+{short[t2]:<20} 평균: {np.mean(scores_2):.4f} ± {np.std(scores_2):.4f}')
print(f'{"─"*70}')

---
## 7. WandB 종료

In [ ]:
if CFG['use_wandb']:
    wandb.finish()
    print('WandB run 종료')